In [ ]:
%matplotlib inline 
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
import multitaper as mt
from scipy import stats

import standardplots
import obs
import run_config

# consistent per-source colors, reused across all comparison plots below
COLORS = run_config.COLORS

In [ ]:
names = obs.NAMES
data = obs.DATAFOLDER

In [ ]:
fig, axes = plt.subplots(1, 1, sharex= True)

standardplots.plot_resolution(axes, 'ch4_EDC', COLORS['EDC'])

In [ ]:
# proxies grouped by what they record (same grouping as the hint in obs.select_data)
GROUPS = {
    'Temperature':          ['ch4_EDC', 'MgCa_Eld'],
    'Continental ice mass': ['dO18_EDC', 'dO18_LR04', 'dO18_Eld'],
    r'CO$_2$':              ['co2_EDC', 'co2_Hon', 'co2_Yam'],
}
LABELS = {
    'ch4_EDC':   r'CH$_4$ (EDC)',
    'MgCa_Eld':  'Mg/Ca (Elderfield 2012)',
    'dO18_EDC':  r'$\delta^{18}$O$_{O_2}$ (EDC)',
    'dO18_LR04': r'benthic $\delta^{18}$O (LR04)',
    'dO18_Eld':  r'seawater $\delta^{18}$O (Elderfield 2012)',
    'co2_EDC':   r'CO$_2$ (EDC)',
    'co2_Hon':   r'CO$_2$ (Hönisch 2009)',
    'co2_Yam':   r'CO$_2$ (Yamamoto 2022)',
}
MARKERS = {'EDC': 'o', 'Eld': 's', 'LR04': '^', 'Hon': 'D', 'Yam': 'v'}
UNITS = {
    'ch4_EDC':   'ppb',
    'MgCa_Eld':  'mmol/mol, normalized',
    'dO18_EDC':  '‰',
    'dO18_LR04': '‰',
    'dO18_Eld':  '‰ SMOW',
    'co2_EDC':   'ppm',
    'co2_Hon':   'ppm',   # Hönisch reports pCO2 in µatm, ~ ppm
    'co2_Yam':   'ppm',
}

# one GPR per record (~4 min total); kept separate from plotting so restyling doesn't refit
gpr = {name: obs.gp_reg(name) for names in GROUPS.values() for name in names}

In [ ]:
from matplotlib.lines import Line2D

AGE, VAR = obs.AGE_COL, obs.VAR_COL

def plot_obs_overview(groups, xmax, standardize=()):
    '''
        groups       -  {panel title: [DATAFOLDER keys]}; insolation is always added as the last panel
        xmax         -  oldest age shown [ka]
        standardize  -  panel titles whose records are z-scored (raw and GPR with the raw record's
                        mean/std), for panels mixing records with different units
    '''
    with plt.rc_context({'font.size': 16}):
        fig, axs = plt.subplots(len(groups) + 1, 1, figsize=(22, 6 * (len(groups) + 1)), sharex=True)

        for ax, (group, names) in zip(axs, groups.items()):
            z = group in standardize
            handles = []
            for name in names:
                src = name.split('_')[1]
                col = obs.NAMES[name]
                full = obs.select_data(name)
                raw = full[full[AGE] <= xmax]  # so the y-range fits only what is shown
                # gp_reg's grid starts at 0 ka; only show the fit where the record has data
                fit = gpr[name][gpr[name][AGE].between(full[AGE].min(), min(full[AGE].max(), xmax))]

                mu, sd = (full[col].mean(), full[col].std()) if z else (0, 1)
                y = (fit[col] - mu) / sd
                s = np.sqrt(fit[VAR].astype(float)) / sd

                ax.scatter(raw[AGE], (raw[col] - mu) / sd, color=COLORS[src], marker=MARKERS[src],
                           s=22, alpha=0.66, linewidths=0, zorder=2)
                ax.fill_between(fit[AGE], y - 2*s, y + 2*s, color=COLORS[src], alpha=0.2, linewidth=0, zorder=1)
                ax.plot(fit[AGE], y, color=COLORS[src], lw=2, zorder=3, alpha = 0.9)
                handles.append(Line2D([], [], color=COLORS[src], marker=MARKERS[src], markersize=9, lw=2,
                                      label=LABELS[name]))

            ax.set_title(group, fontweight='bold')
            ax.set_ylabel(r'standardized anomaly [$\sigma$]' if z
                          else f"{LABELS[names[0]].split(' (')[0]} [{UNITS[names[0]]}]")
            ax.legend(handles=handles, loc='upper left', ncol=len(handles), fontsize=14)
            ax.grid(True, alpha=0.3)

        insol = obs.select_data('insol')
        insol = insol[insol[AGE] <= xmax]
        axs[-1].plot(insol[AGE], insol[obs.NAMES['insol']], color=COLORS['insol'], lw=1.5)
        axs[-1].set_title('Insolation (65°N integrated, Huybers 2006)', fontweight='bold')
        axs[-1].set_ylabel(r'ISI, $\tau$ = 0 [GJ m$^{-2}$]')
        axs[-1].grid(True, alpha=0.3)

        axs[-1].set_xlim(xmax, 0)  # oldest on the left, as in standardplots.plot_rollout
        axs[-1].set_xlabel('Age [ka BP]')
        fig.text(0.99, 0.005, r'points: raw proxy data   lines: GPR mean   bands: GPR $\pm 2\sigma$',
                 ha='right', va='bottom', fontsize=14, color='dimgray')
        plt.tight_layout(rect=[0, 0.015, 1, 1])
    return fig

# temperature / ice panels mix units -> z-scored; CO2 records share ppm
# xmax = 2100 ka covers every proxy except the older part of LR04 (to 5320 ka)
fig = plot_obs_overview(GROUPS, xmax=2100, standardize=('Temperature', 'Continental ice mass'))
#fig.savefig(f"{PROJECT}/Data/Plots/obs_gpr_overview.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# only the records used as emulator state variables (run_config.STATE_COLS); one record per panel,
# so everything stays in native units. xmax = 1600 ka covers Mg/Ca (to 1546 ka) and Yamamoto (to 1461 ka)
fig = plot_obs_overview({'Temperature':          ['MgCa_Eld'],
                         'Continental ice mass': ['dO18_LR04'],
                         r'CO$_2$':              ['co2_Yam']}, xmax=1600)
#fig.savefig(f"{PROJECT}/Data/Plots/obs_gpr_state_vars.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# multitaper spectra of CH4 and Mg/Ca (standardplots.multitaper_spectrum). The function needs an evenly
# spaced series, so it runs on the 1 ka GPR mean (gpr cell above), cut to each record's own age range;
# above ~1/(GPR length scale) the spectrum reflects the GPR smoothing more than the data.
# Both series are z-scored so their power shares one axis.
ORBITAL = {100: 'darkblue', 41: 'slateblue', 23: 'mediumpurple'}  # period [kyr] -> line color

def inv(v):
    with np.errstate(divide='ignore'):
        return 1 / np.asarray(v, dtype=float)

spectra = {}
with plt.rc_context({'font.size': 16}):
    fig, ax = plt.subplots(figsize=(18, 9))

    for name in ['ch4_EDC', 'MgCa_Eld']:
        src = name.split('_')[1]
        raw = obs.select_data(name)
        fit = gpr[name][gpr[name][AGE].between(raw[AGE].min(), raw[AGE].max())]
        x = fit[obs.NAMES[name]].to_numpy()
        x = (x - x.mean()) / x.std()

        freq, spec, ci_lower, ci_upper, significant = standardplots.multitaper_spectrum(x, dt=1, nw=2)
        spectra[name] = {"freq": freq, "spec": spec, "ci_lower": ci_lower, "ci_upper": ci_upper, "significant": significant}

        pos = freq > 0  # f = 0 can't go on a log axis
        sig = pos & significant
        ax.plot(freq[pos], spec[pos], color=COLORS[src], lw=2,
                label=f"{LABELS[name]}, {fit[AGE].min():.0f}–{fit[AGE].max():.0f} ka")
        ax.fill_between(freq[pos], ci_lower[pos], ci_upper[pos], color=COLORS[src], alpha=0.15, linewidth=0)
        ax.scatter(freq[sig], spec[sig], color=COLORS[src], marker=MARKERS[src], edgecolor='black', s=45, zorder=5)

    for period, color in ORBITAL.items():
        ax.axvline(1/period, color=color, linestyle='dashed', alpha=0.6, label=f'{period} kyr cycle')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('frequency [1/kyr]')
    ax.set_ylabel(r'power [$\sigma^2$ kyr]')
    ax.secondary_xaxis('top', functions=(inv, inv)).set_xlabel('period [kyr]')
    ax.set_title('Multitaper spectra: CH$_4$ vs. Mg/Ca (temperature proxies)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower left', fontsize=14)
    fig.text(0.99, 0.005, r'bands: 95% $\chi^2$ CI   markers: Thomson F-test significant at 95%',
             ha='right', va='bottom', fontsize=14, color='dimgray')
    plt.tight_layout(rect=[0, 0.02, 1, 1])
    #fig.savefig(f"{PROJECT}/Data/Plots/spectra_ch4_mgca.pdf", bbox_inches='tight')
    plt.show()